# FieldLens ML 02: Train and convert

**Goal (Day 12):** a `.tflite` file that is small, fast and accurate enough.

Plan:
1. Set up and load the dataset (same split as Day 11)
2. Train the head of a frozen MobileNetV3-Small (stage 1)
3. Unfreeze the top blocks and fine-tune gently (stage 2)
4. Evaluate on the test set: per-class precision and recall, confusion matrix
5. Convert to TFLite in three variants: float32, float16, full int8
6. Verify every converted model still predicts correctly
7. Write `model_card.md` and save everything to Drive

**Rules we follow**
- The **val** set is used for early stopping. The **test** set is used **once**, at the end.
- Augmentation is applied to **train only**.
- The representative dataset for int8 comes from the **train** split, never test.

## Step 1: Setup

In [ ]:
import random
import shutil
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU. Runtime > Change runtime type > T4 GPU"

Load the dataset from Drive (same zip as Day 11). Colab's disk is wiped when the
runtime resets, so this cell may need running again in a new session.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/fieldlens/processed.zip"
!rm -rf /content/data && mkdir -p /content/data
!unzip -q "$ZIP_PATH" -d /content/data

root = next(Path("/content/data").rglob("manifest.csv")).parent
print("Dataset root:", root)

## Step 2: Data pipeline

Images are resized to 224x224 (bilinear, no cropping). Augmentation is applied only
to the training set, **after** the split, so a transformed copy can never leak into val or test.

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def load(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        root / split,
        image_size=IMG_SIZE,
        batch_size=BATCH,
        shuffle=shuffle,
        seed=SEED,
        label_mode="int",
    )


train_raw = load("train", shuffle=True)
val_ds = load("val", shuffle=False)
test_ds = load("test", shuffle=False)

class_names = train_raw.class_names  # alphabetical: this order is part of the model contract
n_classes = len(class_names)
print("Class order:", class_names)

# Class weights from the train split only: total / (n_classes * count).
train_counts = {
    c: sum(1 for f in (root / "train" / c).iterdir() if f.suffix.lower() in IMAGE_SUFFIXES)
    for c in class_names
}
total = sum(train_counts.values())
class_weight = {i: total / (n_classes * train_counts[c]) for i, c in enumerate(class_names)}
for i, c in enumerate(class_names):
    print(f"{c:8} train={train_counts[c]:5} weight={class_weight[i]:.2f}")

augment = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
    ],
    name="augment",
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

Quick look: one training image, then seven augmented versions of it.

In [ ]:
for images, _ in train_raw.take(1):
    sample = images[0]

plt.figure(figsize=(12, 3))
for i in range(8):
    shown = sample if i == 0 else augment(sample[None], training=True)[0]
    plt.subplot(1, 8, i + 1)
    plt.imshow(np.clip(shown.numpy(), 0, 255).astype("uint8"))
    plt.title("original" if i == 0 else f"aug {i}", fontsize=8)
    plt.axis("off")
plt.show()

## Step 3: Build the model

MobileNetV3-Small, pretrained on ImageNet, with a new 5-class head.

`include_preprocessing=True` means the model scales the pixels **inside itself**. It expects raw
RGB values 0-255. This makes Day 14 simpler: the phone only has to resize and pass raw pixels.

In [ ]:
def build_model():
    base = tf.keras.applications.MobileNetV3Small(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet",
        pooling="avg",
        include_preprocessing=True,
    )
    base.trainable = False  # stage 1: frozen

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,), name="image")
    x = base(inputs, training=False)  # keeps BatchNorm in inference mode, also later
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(n_classes, activation="softmax", name="probs")(x)
    return tf.keras.Model(inputs, outputs, name="fieldlens_mnv3s"), base


model, base = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Trainable parameters:", int(sum(np.prod(v.shape) for v in model.trainable_weights)))
print("Total parameters:    ", model.count_params())

## Step 4: Stage 1, train the head

Only the new final layer learns. EarlyStopping watches **val loss** and restores the best epoch.
`class_weight` makes mistakes on small classes (METAL, PLASTIC) cost more.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=[early_stop],
)
print("Stage 1 best val accuracy:", max(hist1.history["val_accuracy"]))

## Step 5: Stage 2, fine-tune the top blocks

Unfreeze the last layers of the base and train them with a **very small learning rate** so
we adjust the pretrained knowledge instead of destroying it. BatchNorm layers stay frozen.

In [ ]:
FINE_TUNE_LAYERS = 40

base.trainable = True
for layer in base.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base.layers[-FINE_TUNE_LAYERS:]:
    layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)

# Changing trainable needs a re-compile.
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Trainable parameters now:", int(sum(np.prod(v.shape) for v in model.trainable_weights)))

early_stop2 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)
hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=[early_stop2],
)
print("Stage 2 best val accuracy:", max(hist2.history["val_accuracy"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for hist, label in [(hist1, "stage 1"), (hist2, "stage 2")]:
    axes[0].plot(hist.history["loss"], label=f"{label} train")
    axes[0].plot(hist.history["val_loss"], "--", label=f"{label} val")
    axes[1].plot(hist.history["accuracy"], label=f"{label} train")
    axes[1].plot(hist.history["val_accuracy"], "--", label=f"{label} val")
axes[0].set_title("loss")
axes[1].set_title("accuracy")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

## Save the trained model now

Colab can reset the runtime at any time and the disk is wiped when it does. Saving to Drive
right after training means a reset can never cost us the 25 minutes of training again.

In [ ]:
DRIVE_OUT = Path("/content/drive/MyDrive/fieldlens/models")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
model.save(DRIVE_OUT / "fieldlens_keras.keras")
print("Saved:", DRIVE_OUT / "fieldlens_keras.keras")

## Step 6: Evaluate on the test set

This is the **one** time we use the test set. One accuracy number can hide a class that never
fires, so we look at precision and recall per class and at the confusion matrix.

In [ ]:
x_test = np.concatenate([x.numpy() for x, _ in test_ds])
y_test = np.concatenate([y.numpy() for _, y in test_ds])
print("Test images:", x_test.shape, "| labels:", y_test.shape)

y_prob = model.predict(x_test, batch_size=64, verbose=0)
y_pred = y_prob.argmax(axis=1)

keras_acc = float((y_pred == y_test).mean())
keras_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Keras model  accuracy: {keras_acc:.3f}  macro-F1: {keras_f1:.3f}")

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(cm, index=[f"true {c}" for c in class_names], columns=[f"pred {c}" for c in class_names]))

plt.figure(figsize=(5, 4.5))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(n_classes), class_names, rotation=45)
plt.yticks(range(n_classes), class_names)
for i in range(n_classes):
    for j in range(n_classes):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("predicted")
plt.ylabel("true")
plt.title("Confusion matrix (test)")
plt.tight_layout()
plt.show()

print("Does any class never fire?")
print(pd.DataFrame({"true": np.bincount(y_test, minlength=n_classes),
                    "predicted": np.bincount(y_pred, minlength=n_classes)}, index=class_names))

## Step 7: Convert to TFLite (four variants)

| Variant | What it is |
| --- | --- |
| float32 | The baseline, no quantization |
| float16 | Weights stored in 16 bits, about half the size |
| int8 | Weights **and activations** in 8 bits, calibrated with a **representative dataset** |
| dynamic_range | Weights in int8, activations stay float at run time. No calibration data needed |

The representative dataset is 1000 **train** images (200 per class, no labels). The converter runs
them through the model to learn the range of the activations.

All variants take and return **float32** at their edges, which is simpler for the app.

In [ ]:
OUT = Path("/content/models")
OUT.mkdir(exist_ok=True)
SAVED = "/content/saved_model"
shutil.rmtree(SAVED, ignore_errors=True)
model.export(SAVED)  # Keras 3 route to TFLite: export a SavedModel, then convert it


def make_rep_data(n_per_class):
    def gen():
        rng = random.Random(SEED)
        files = []
        for c in class_names:
            class_files = sorted(f for f in (root / "train" / c).iterdir() if f.suffix.lower() in IMAGE_SUFFIXES)
            files += rng.sample(class_files, n_per_class)
        rng.shuffle(files)
        for f in files:
            img = tf.io.decode_image(tf.io.read_file(str(f)), channels=3, expand_animations=False)
            img = tf.image.resize(img, IMG_SIZE)  # same resize as the training pipeline
            yield [img[None].numpy().astype("float32")]
    return gen


def convert(name, configure=None):
    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED)
    if configure:
        configure(converter)
    blob = converter.convert()
    path = OUT / f"fieldlens_{name}.tflite"
    path.write_bytes(blob)
    print(f"{name:14} {len(blob) / 1e6:6.2f} MB -> {path.name}")
    return path


def cfg_float16(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]


def cfg_int8(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = make_rep_data(200)
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]


def cfg_dynamic(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]


paths = {
    "float32": convert("float32"),
    "float16": convert("float16", cfg_float16),
    "int8": convert("int8", cfg_int8),
    "dynamic_range": convert("dynamic", cfg_dynamic),
}

for f in OUT.iterdir():  # checkpoint to Drive straight away
    shutil.copy(f, DRIVE_OUT / f.name)
print("Copied to Drive:", sorted(p.name for p in OUT.iterdir()))

## Step 8: Verify every converted model

Conversion can silently degrade a model. Each `.tflite` runs on the **whole test set**. We compare
it with the Keras model: accuracy, METAL recall, and how often both pick the same class.

TFLite's default CPU delegate is **XNNPACK**. If it fails to load a file, the cell says so and
retries without it. That failure is itself a result: the phone uses the same default.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow.lite.python.interpreter")

try:
    from ai_edge_litert.interpreter import Interpreter, OpResolverType  # LiteRT, the new name
    print("Using the LiteRT interpreter")
except ImportError:
    Interpreter = tf.lite.Interpreter
    OpResolverType = tf.lite.experimental.OpResolverType
    print("Using tf.lite.Interpreter")


def make_interpreter(path, use_xnnpack):
    kwargs = {}
    if not use_xnnpack:
        kwargs["experimental_op_resolver_type"] = OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    interp = Interpreter(model_path=str(path), **kwargs)
    interp.allocate_tensors()
    return interp


def run_tflite(path, x):
    cpu_path = "default (XNNPACK)"
    try:
        interp = make_interpreter(path, use_xnnpack=True)
    except RuntimeError as err:
        print(f"!! {path.name}: default CPU delegate failed: {str(err)[:120]}")
        interp = make_interpreter(path, use_xnnpack=False)
        cpu_path = "no XNNPACK (it failed)"
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    probs = np.zeros((len(x), out["shape"][-1]), dtype=np.float32)
    for i in range(len(x)):
        sample = x[i : i + 1]
        if inp["dtype"] != np.float32:
            scale, zero = inp["quantization"]
            info = np.iinfo(inp["dtype"])
            sample = np.clip(np.round(sample / scale + zero), info.min, info.max).astype(inp["dtype"])
        interp.set_tensor(inp["index"], sample)
        interp.invoke()
        res = interp.get_tensor(out["index"])
        if out["dtype"] != np.float32:
            scale, zero = out["quantization"]
            res = (res.astype(np.float32) - zero) * scale
        probs[i] = res[0]
    return probs, inp, out, cpu_path


metal = class_names.index("METAL")


def score(name, path, probs, cpu_path):
    pred = probs.argmax(axis=1)
    return {
        "variant": name,
        "size_MB": round(path.stat().st_size / 1e6, 2),
        "accuracy": round(float((pred == y_test).mean()), 4),
        "macro_F1": round(float(f1_score(y_test, pred, average="macro")), 4),
        "METAL_recall": round(float(((pred == metal) & (y_test == metal)).sum() / (y_test == metal).sum()), 3),
        "same_class_as_keras": round(float((pred == y_pred).mean()), 4),
        "default_cpu_delegate": "ok" if cpu_path.startswith("default") else "FAILED",
    }


rows = []
for name, path in paths.items():
    print("Running", path.name, "...")
    probs, inp, out, cpu_path = run_tflite(path, x_test)
    rows.append(score(name, path, probs, cpu_path))
rows.append({"variant": "keras (reference)", "size_MB": None, "accuracy": round(keras_acc, 4),
             "macro_F1": round(keras_f1, 4), "METAL_recall": None, "same_class_as_keras": 1.0,
             "default_cpu_delegate": None})
results = pd.DataFrame(rows)
print(results.to_string(index=False))

## Step 8b: Why is full int8 so much worse? (investigation, optional to re-run)

On the first run, full int8 lost about 10 points of accuracy while float16 lost 0.1. Before accepting
that, we tried to fix it. Results from that run (test set, 984 images):

| Experiment | Size MB | Accuracy | Macro-F1 | METAL recall | Same class as Keras | Default delegate |
| --- | ---: | ---: | ---: | ---: | ---: | --- |
| A: int8 strict, 300 calibration images | 1.24 | 0.8120 | 0.8004 | 0.835 | 0.8394 | failed |
| B: int8 strict, 1000 calibration images | 1.24 | 0.8079 | 0.7876 | 0.896 | 0.8354 | ok |
| C: int8 with float fallback, 300 images | 1.24 | 0.8120 | 0.8004 | 0.835 | 0.8394 | failed |
| D: int8 with float fallback, 1000 images | 1.24 | 0.8079 | 0.7876 | 0.896 | 0.8354 | ok |
| E: dynamic range | 1.13 | 0.9126 | 0.9012 | 0.930 | 0.9675 | ok |

What we learned:
- Allowing a float fallback changed nothing (A = C, B = D): every op was already quantized.
- More calibration images did not help (300 to 1000). So this is not a calibration-size problem.
  The likely cause is that MobileNetV3's hard-swish and squeeze-excite blocks are sensitive to
  int8 (this is a hypothesis; a quantization debugger would pinpoint the layers).
- The default XNNPACK delegate failed on the 300-image files but loaded the 1000-image files.
  Always test the exact file with the default delegate on a real device.
- Dynamic range keeps accuracy within 0.5 points of float32 and is the smallest file.

The cell below re-runs the experiments (about 5 to 8 minutes).

In [ ]:
EXP = Path("/content/exp")
EXP.mkdir(exist_ok=True)


def try_variant(name, n_per_class=None, strict=True, dynamic=False):
    conv = tf.lite.TFLiteConverter.from_saved_model(SAVED)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if not dynamic:
        conv.representative_dataset = make_rep_data(n_per_class)
        if strict:
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    path = EXP / f"{name}.tflite"
    path.write_bytes(conv.convert())
    probs, _, _, cpu_path = run_tflite(path, x_test)
    return score(name, path, probs, cpu_path)


experiments = [
    ("A_strict_300", dict(n_per_class=60, strict=True)),
    ("B_strict_1000", dict(n_per_class=200, strict=True)),
    ("C_fallback_300", dict(n_per_class=60, strict=False)),
    ("D_fallback_1000", dict(n_per_class=200, strict=False)),
    ("E_dynamic_range", dict(dynamic=True)),
]
exp_rows = []
for name, kwargs in experiments:
    print("Running", name, "...")
    try:
        exp_rows.append(try_variant(name, **kwargs))
    except Exception as err:
        print(f"!! {name} failed: {str(err)[:200]}")
print(pd.DataFrame(exp_rows).to_string(index=False))

## Step 9: Write `model_card.md` and save everything

The model card records exactly how images must be prepared. On Day 14 the phone must do the
same thing, otherwise the model "works in Python but not on the phone".

In [ ]:
def md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |")
    return "\n".join(lines)


test_counts = np.bincount(y_test, minlength=n_classes)
report_text = classification_report(y_test, y_pred, target_names=class_names, digits=3)
card = [
    "# Model card: FieldLens waste classifier",
    "",
    "## Model",
    "- Architecture: MobileNetV3-Small, ImageNet-pretrained, new 5-class softmax head",
    "- Training: stage 1 trains the head (frozen base, lr 1e-3); stage 2 fine-tunes the last "
    f"{FINE_TUNE_LAYERS} layers (BatchNorm frozen, lr 1e-5). Class weights, early stopping on val loss.",
    "- Files: `fieldlens_float32.tflite`, `fieldlens_float16.tflite`, `fieldlens_int8.tflite`, "
    "`fieldlens_dynamic.tflite`",
    "",
    "## Variants and results (test set, 984 images)",
    md_table(results),
    "",
    "- **float32**: baseline, no quantization.",
    "- **float16**: weights stored in 16 bits.",
    "- **int8**: full integer quantization (weights and activations), calibrated with a representative "
    "dataset of 1000 train images (200 per class). Input and output stay float32.",
    "- **dynamic_range**: weights in int8, activations stay float at run time. No calibration data needed.",
    "",
    "**Recommendation:** the full int8 variant is NOT recommended for production. It lost about 10 accuracy "
    "points and agrees with float32 on only about 84% of images. The candidates for production are float16 "
    "and dynamic_range (within about 0.5 points of float32). The final choice is made after the Day 15 "
    "on-device benchmark and recorded in an ADR.",
    "",
    "## Input (the contract the app must follow)",
    "- Tensor shape: `[1, 224, 224, 3]`, dtype `float32` (all four files)",
    "- Channel order: **RGB**",
    "- Pixel values: **raw 0 to 255** as floats. Do NOT divide by 255 and do NOT subtract a mean.",
    "  Scaling is done inside the model (`include_preprocessing=True`).",
    "- Resize: straight to 224x224, **bilinear**, **no cropping and no aspect-ratio padding** (the image is stretched).",
    "- Test the app by running one image through Python and Dart and comparing the output tensors.",
    "",
    "## Output",
    "- Shape `[1, 5]`, float32, **softmax probabilities** (sum to 1).",
    f"- Class order (index to label): {', '.join(f'{i}={c}' for i, c in enumerate(class_names))}",
    "- There is no 'unknown' class: the model always picks one of the five. "
    "The app should treat low confidence as `uncertain`.",
    "",
    "## Training data",
    "- Kaggle 'Garbage Classification' (ODbL), 8 source folders mapped to 5 classes (see `docs/dataset.md`).",
    f"- Split (group-aware, seed 42): {sum(train_counts.values())} train / 986 val / {len(y_test)} test.",
    f"- Train counts: {train_counts}. Class weights: { {class_names[i]: round(w, 2) for i, w in class_weight.items()} }",
    "- Augmentation (train only): horizontal flip, rotation, zoom, brightness.",
    "",
    "## Per-class results (float32, test set)",
    "```",
    report_text,
    "```",
    "",
    "## Known limitations",
    "- Only 5 classes. Anything else (clothes, batteries, mixed items) still gets one of the five.",
    "- GLASS is often confused with PLASTIC and METAL: shiny, transparent containers. "
    "Source images are web-scraped and some label noise remains.",
    f"- Small test sets: METAL has only {int(test_counts[metal])} test images, so its recall is uncertain by about +/-5 points.",
    "- Many photos are single objects on plain backgrounds. Messy, real-world scenes may score lower.",
    "- Post-training full int8 hurts this model (MobileNetV3 hard-swish and squeeze-excite blocks are "
    "quantization-sensitive; more calibration images did not help). Possible fixes: quantization-aware "
    "training, a quantization-friendly backbone (EfficientNet-Lite0, minimalistic MobileNetV3).",
    "- The int8 file calibrated with 300 images failed to load with TFLite's default XNNPACK delegate; the "
    "1000-image version loaded. Always test the exact file with the default delegate on a real device.",
    "- Accuracy is measured with TensorFlow image decoding. The phone's decoder may differ very slightly.",
]
(OUT / "model_card.md").write_text("\n".join(card), encoding="utf-8")
results.to_csv(OUT / "results.csv", index=False)

for f in OUT.iterdir():
    shutil.copy(f, DRIVE_OUT / f.name)
print("Saved to Drive:", sorted(p.name for p in DRIVE_OUT.iterdir()))

## Next

1. Download the `.tflite` files, `model_card.md` and `results.csv` from `MyDrive/fieldlens/models/` into `apps/ml/models/` in the repo (skip the `.keras` file).
2. Read `model_card.md` and fix anything that does not match what you saw.
3. Commit the notebook, the three `.tflite` files and `model_card.md`.